In [ ]:
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
def generate_synthetic_dataset(n=1000):
  sex = np.random.choice(['M', 'F'], size=n, p=[0.5, 0.5])
  age = np.random.randint(18, 71, size=n)

  height = np.empty(n)
  weight = np.empty(n)
  bf_percentage = np.empty(n)
  skinfold_thickness = np.empty(n)

  for i in range(n):
    if sex[i] == "M":
      height[i] = np.random.normal(175, 7)
      weight[i] = np.random.normal(82, 12)
      bf_percentage[i] = np.random.normal(20, 5)
    else:
      height[i] = np.random.normal(162, 6.5)
      weight[i] = np.random.normal(68, 11)
      bf_percentage[i] = np.random.normal(28, 6)

    height[i] = round(max(140, min(210, height[i])), 1)
    weight[i] = round(max(40, min(160, weight[i])), 1)
    bf_percentage[i] = round(max(8, min(50, bf_percentage[i])), 1)

    skinfold_thickness[i] = bf_percentage[i] * 0.8 + np.random.normal(0, 3)
    skinfold_thickness[i] = round(max(5, min(45, skinfold_thickness[i])))

  df = pd.DataFrame(
      {
          'id': np.arange(n),
          'age': age,
          'sex': sex,
          'height_cm': height,
          'weight_kg': weight,
          'body_fat_percentage': bf_percentage,
          'skinfold_thickness_mm': skinfold_thickness
      }
  )
  return df

In [ ]:
def corrupt_dataset(df, corruption_rate=0.03):
  dirty_df = df.copy()
  corruptions = []

  n_rows = len(df)
  for idx in range(n_rows):
    if np.random.rand() > corruption_rate:
      continue

    error_type = np.random.choice([
        'height_units',
        'weight_units',
        'transposed_digits',
        'missing_decimal',
        'height_and_weight_swap',
        'sex_swap'
    ])

    original_val = None
    new_val = None
    col = None

    if error_type == 'height_units':
      col = 'height_cm'
      original_val = dirty_df.at[idx, col]
      new_val = round(original_val / 2.54, 1)
      dirty_df.at[idx, col] = new_val

    elif error_type == 'weight_units':
      col = 'weight_kg'
      original_val = dirty_df.at[idx, col]
      new_val = round(original_val * 2.20462, 1)
      dirty_df.at[idx, col] = new_val

    elif error_type == 'missing_decimal':
      col = np.random.choice(['height_cm', 'weight_kg', 'body_fat_percentage', 'skinfold_thickness_mm'])
      original_val = dirty_df.at[idx, col]
      new_val = float(str(original_val).replace('.', ''))
      dirty_df.at[idx, col] = new_val

    elif error_type == 'transposed_digits':
      col = np.random.choice(['age', 'height_cm', 'weight_kg', 'body_fat_percentage', 'skinfold_thickness_mm'])
      original_val = dirty_df.at[idx, col]
      val_str = list(str(original_val))

      if len(val_str) >= 2:
        swap_i = 0 if val_str[1] != '.' else 2
        if swap_i + 1 < len(val_str):
          val_str[swap_i], val_str[swap_i + 1] = val_str[swap_i + 1], val_str[swap_i]
          new_val = float("".join(val_str))
          dirty_df.at[idx, col] = new_val

    elif error_type == 'height_and_weight_swap':
      col = 'height_cm & weight_kg'
      original_val = f'height: {dirty_df.at[idx, 'height_cm']}, weight: {dirty_df.at[idx, 'weight_kg']}'

      temp = dirty_df.at[idx, 'height_cm']
      dirty_df.at[idx, 'height_cm'] = dirty_df.at[idx, 'weight_kg']
      dirty_df.at[idx, 'weight_kg'] = temp

      new_val = f'height: {dirty_df.at[idx, 'height_cm']}, weight: {dirty_df.at[idx, 'weight_kg']}'

    elif error_type == 'sex_swap':
      col = 'sex'
      original_val = dirty_df.at[idx, 'sex']
      new_val = 'M' if original_val == 'F' else 'F'
      dirty_df.at[idx, 'sex'] = new_val

    if col and new_val is not None:
      corruptions.append({
          'id': dirty_df.at[idx, 'id'],
          'column(s)': col,
          'error_type': error_type,
          'original_value': original_val,
          'corrupted_value': new_val
      })

  answer_key_df = pd.DataFrame(corruptions)
  return dirty_df, answer_key_df

In [ ]:
df = generate_synthetic_dataset()
dirty_df, answer_key_df = corrupt_dataset(df)

answer_key_df

,id,column(s),error_type,original_value,corrupted_value
0,46,weight_kg,weight_units,79.5,175.3
1,97,height_cm,transposed_digits,175.6,715.6
2,135,weight_kg,weight_units,82.4,181.7
3,237,height_cm,height_units,178.8,70.4
4,254,weight_kg,weight_units,77.6,171.1
5,305,weight_kg,weight_units,45.6,100.5
6,366,height_cm,height_units,165.6,65.2
7,402,height_cm,height_units,170.8,67.2
8,415,skinfold_thickness_mm,missing_decimal,19.0,190.0
9,438,weight_kg,transposed_digits,64.4,46.4


In [ ]:
dirty_df.to_csv('easter_egg_hunt.csv', index=False)
answer_key_df.to_csv('answer_key.csv', index=False)